<div style="text-align: center;">

<h1>BattleActs Actuarial Exam Preparation Products</h1>
<h3>DATASCI 530 — Project 1</h3>
<p>Maxine Hakimi-Bulson</p>

</div>

## Introduction

BattleActs is an online education and exam-prep platform that helps students pass professional actuarial exams for the Casualty Actuarial Society (CAS). CAS is a professional organization that administers credentialing exams for actuaries specializing in property and casualty insurance, such as auto and homeowners insurance. An actuary uses mathematics, statistics, and financial models to assess risk and estimate the costs of uncertain events. Used by more than 2,000 students, BattleActs offers study guides on six actuarial exams, each comprising of articles, quiz banks, infinite calculation problems, and other tools made to help students study for (and hopefully pass) their actuary exams.

For this project, I collected information about BattleActs' six study guides for Exams 5, 6C, 6U, 7, 8, and 9 from their website. I extracted each course's price, access period, and advertised study tool descriptions and features. I also collected the publicly available PDF and Excel sample study materials for each exam in order to compare the content, length, and detail of materials offered for each exam.

All product information and sample materials came from [BattleActs](https://battleactsmain.ca/v2/home). The company granted written permission for this one-time educational web scraping project and for sharing the code, data, and analysis on GitHub. Data were collected on October 19, 2026. Collection was limited to public pages and linked PDF and Excel files. No accounts, paid course materials, customer information, or video downloads were collected. I included pauses between requests.

## Data Collection Process

Before writing the web-scraping algorithm and code, I manually collected data on two exam preparation guides and recorded their information in a spreadsheet. This helped me determine which information I wanted to explore and do future analysis in the project, and how and where to locate such data on the website. The script opens the Exams menu at the top of the home page, and uses a loop to click through all six course pages, collect the product information, and extract URLs. Each unique PDF or Excel file is downloaded once and saved in folders organized by file type and exam.

The collected information is organized into three datasets:

- **Products:** one row per preparation course, including prices, descriptions, features, access periods, and what makes that exam unique.

- **PDF Samples:** one row per unique PDF, including its sample label, source URL, page count, and extracted text.

- **Excel Samples:** one row per unique workbook, including its sample label, source URL, worksheet names and counts, formula count, and extracted cell contents.

When multiple sample cards link to the same file, their labels are combined in one record. Text is extracted from every PDF page and every Excel worksheet, and Excel formulas are retained as expressions rather than calculated results.

## Import Libraries for Web Scraping

In [5]:
import re
import time
from pathlib import Path
from urllib.parse import unquote, urlsplit

import pandas as pd
import requests
from IPython.display import HTML, display
from openpyxl import load_workbook
from pypdf import PdfReader
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

START_URL = "https://battleactsmain.ca/v2/home"
STYLE = ("<style>.report-table table{border-collapse:collapse;width:100%;font-size:13px}"
         ".report-table th{background:#eaf0f7;color:#183e60;text-align:left}"
         ".report-table th,.report-table td{padding:9px;border-bottom:1px solid #dce3eb;white-space:normal}"
         ".report-table tr:nth-child(even){background:#f7f9fb}</style>")


def now():
    return pd.Timestamp.now(tz="America/New_York").isoformat(timespec="seconds")


def show_table(frame):
    html = frame.to_html(index=False, na_rep="—", border=0)
    display(HTML(STYLE + '<div class="report-table">' + html + "</div>"))

## Product Extraction Function

In [2]:
def extract_product(driver):
    """Extract one product's details and its sample cards from the open page."""
    def text(xpath):
        found = driver.find_elements("xpath", xpath)
        return found[0].text.strip() if found else None

    def join(xpath):
        return "\n".join(e.text.strip() for e in driver.find_elements("xpath", xpath)) or None

    def number(pattern, source):
        match = re.search(pattern, source or "", re.I)
        return int(match.group(1)) if match else None

    guide = '//h2[contains(., "Study Guide")]/parent::div'
    price_text = text('//h2[contains(., "$")]')
    features = join('//h2[normalize-space()="Overview Of Our Features"]/parent::div/parent::div//ul/li')
    name, url, stamp = text("//h1"), driver.current_url, now()

    product = {
        "product_name": name,
        "price": float(price_text.split()[0].replace("$", "").replace(",", "")) if price_text else None,
        "currency": price_text.split()[-1] if price_text else None,
        "description": join(f"{guide}/p | {guide}//li"),
        "features": features,
        "duration_months": number(r"(\d+)-month'?s?\s+access", features),
        "product_url": url,
        "collection_date": stamp,
        "exam_differences": text('//p[contains(., "different than other")]/following-sibling::p[1]'),
        "practice_exam_count": number(r"(\d+)\s+original practice exams?", features),
        "free_extension_months": number(r"free\s+(\d+)-month\s+extension", features),
    }

    has_class = 'contains(concat(" ", normalize-space(@class), " "), " {} ")'
    cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        f'/ancestor::div[{has_class.format("product-section")}][1]'
        f'//div[{has_class.format("owl-item")} and not({has_class.format("cloned")})]')

    kinds = {".pdf": "PDF", ".xlsx": "Excel", ".xls": "Excel", ".mp4": "Video"}
    samples = []
    for card in cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements("xpath", ".//a[@href] | .//button[@data-url]")
        if not labels or not controls:
            raise ValueError("Unexpected sample-card structure on " + url)
        sample_url = controls[0].get_attribute("href") or controls[0].get_attribute("data-url")
        suffix = Path(unquote(urlsplit(sample_url or "").path)).suffix.lower()
        samples.append({
            "product_name": name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": controls[0].get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": kinds.get(suffix, "Unknown"),
            "product_url": url,
            "collection_date": stamp,
        })
    return product, samples

## Automated Navigation and Collection

In [3]:
driver = webdriver.Chrome(options=webdriver.ChromeOptions())
wait = WebDriverWait(driver, 15)
menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
item_xpath = '//a[contains(@class, "dropdown-item") and contains(@href, "/v2/product/")]'


def open_exam_menu():
    driver.get(START_URL)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()


# Collect the six product URLs (duplicates removed, order kept)
open_exam_menu()
exam_urls = list(dict.fromkeys(a.get_attribute("href") for a in driver.find_elements("xpath", item_xpath)))
if len(exam_urls) != 6:
    raise ValueError(f"Expected 6 exam links, found {len(exam_urls)}. Check the menu.")

# Click each product link in turn and extract that page
products, all_samples = [], []
for exam_url in exam_urls:
    open_exam_menu()
    link = f'//a[contains(@class, "dropdown-item") and @href="{exam_url}"]'
    wait.until(EC.element_to_be_clickable((By.XPATH, link))).click()
    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # let the sample carousel initialize
    product, samples = extract_product(driver)
    products.append(product)
    all_samples.extend(samples)
    time.sleep(3)  # pause between products

driver.quit()
products_df, samples_df = pd.DataFrame(products), pd.DataFrame(all_samples)

## Prepare Study-Tool Indicators

In [24]:
# Flag tools mentioned in the description, feature list, or any sample label
labels = samples_df.groupby("product_url")["sample_label"].agg("\n".join)
tool_text = (products_df["description"].fillna("") + "\n" + products_df["features"].fillna("")
             + "\n" + products_df["product_url"].map(labels).fillna(""))
tool_patterns = {
    "instructional_video_mentioned": r"video lessons|instructional videos?",
    "formula_summary_sheet_mentioned": r"formula|summary sheets?",
    "hybrid_exam_mentioned": r"hybrid CAS exam",
}
products_final = products_df.copy()

for column, pattern in tool_patterns.items():
    products_final[column] = tool_text.str.contains(
        pattern, case=False, na=False
    )

## Download and Extract PDF and Excel Samples

In [5]:
def read_pdf(path):
    reader = PdfReader(path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text, {"page_count": len(reader.pages)}


def read_xlsx(path):
    book = load_workbook(path, read_only=True, data_only=False)
    try:
        lines, formulas = [], 0

        for sheet in book.worksheets:
            lines.append(f"Worksheet: {sheet.title}")

            for row in sheet.iter_rows():
                cells = [c for c in row if c.value is not None]
                formulas += sum(c.data_type == "f" for c in cells)

                if cells:
                    lines.append(" | ".join(str(c.value) for c in cells))

        return "\n".join(lines), {
            "sheet_count": len(book.sheetnames),
            "sheet_names": " | ".join(book.sheetnames),
            "formula_cells": formulas
        }
    finally:
        book.close()


readers = {"PDF": read_pdf, "Excel": read_xlsx}

# Include linked PDFs and Excel files; exclude videos and missing URLs
linked = samples_df[
    samples_df["resource_type"].isin(readers)
    & samples_df["sample_url"].fillna("").str.strip().ne("")
]

records = []

for url, group in linked.groupby("sample_url", sort=False):
    kind = group["resource_type"].iloc[0]
    product_name = group["product_name"].iloc[0]
    exam = re.sub(r"[^A-Za-z0-9]+", "_", product_name).strip("_")
    label = group["sample_label"].iloc[0].lower()

    if "study manual" in label:
        filename = "Study_manual"
    elif "formula" in label or "summary sheet" in label:
        filename = "Formula_summary_sheet"
    elif "past cas exams" in label:
        filename = "Past_CAS_exams"
    else:
        filename = "Practice_problems_and_exams"

    suffix = Path(unquote(urlsplit(url).path)).suffix.lower()
    path = Path("sample_files") / kind / exam / f"{filename}{suffix}"
    path.parent.mkdir(parents=True, exist_ok=True)

    record = {
        "sample_url": url,
        "content_collection_date": now(),
        "extraction_status": "Failed"
    }

    try:
        if not path.exists():
            response = requests.get(
                url,
                headers={"User-Agent": "DATASCI530 educational project"},
                timeout=60
            )
            response.raise_for_status()
            path.write_bytes(response.content)
            time.sleep(3)

        text, details = readers[kind](path)
        record.update(details, extracted_text=text.strip() or None)
        record["extraction_status"] = (
            "Extracted" if text.strip() else "No text extracted"
        )

    except Exception as error:
        record["extraction_status"] = f"Failed: {type(error).__name__}"

    records.append(record)

content_df = pd.DataFrame(records)

## Build and Check the Three Final Datasets

In [3]:
# One record per unique file; labels of cards that share a file are kept together
files_final = linked.groupby("sample_url", sort=False).agg(
    product_name=("product_name", "first"),
    product_url=("product_url", "first"),
    resource_type=("resource_type", "first"),
    collection_date=("collection_date", "first"),
    sample_label=("sample_label", lambda s: " | ".join(dict.fromkeys(s))),
).reset_index().merge(content_df, on="sample_url", how="left", validate="one_to_one")

common = ["product_name", "sample_label", "sample_url", "product_url", "collection_date",
          "content_collection_date", "extraction_status", "extracted_text"]
pdf_samples = files_final.loc[
    files_final["resource_type"].eq("PDF")
].reindex(columns=common + ["page_count"]).copy()
excel_samples = files_final.loc[
    files_final["resource_type"].eq("Excel")
].reindex(
    columns=common + ["sheet_count", "sheet_names", "formula_cells"]
).copy()

# Quality checks
assert len(products_final) == 6 and products_final["product_url"].is_unique
assert products_final[["price", "currency", "duration_months", "practice_exam_count"]].notna().all().all()
assert len(pdf_samples) + len(excel_samples) == linked["sample_url"].nunique()
for frame in (pdf_samples, excel_samples):
    assert frame["sample_url"].is_unique
    assert frame["product_url"].isin(products_final["product_url"]).all()

NameError: name 'linked' is not defined

In [7]:
#Store the three datasets for results tables and analysis
products_final = pd.read_csv("battleacts_products.csv")
pdf_samples = pd.read_csv("battleacts_pdf_samples.csv")
excel_samples = pd.read_csv("battleacts_excel_samples.csv")

## Results

### Table 1. Exam Preparation Course Overview

Prices are shown in their original currencies. The extension is conditional; seven months of standard access should not be read as thirteen months of guaranteed access.

In [23]:
overview = products_final[["product_name", "price", "currency", "duration_months",
                           "practice_exam_count", "free_extension_months"]].rename(columns={
    "product_name": "Exam Prep Course", "price": "Listed Price", "currency": "Currency",
    "duration_months": "Access (Months)", "practice_exam_count": "Original Practice Exams",
    "free_extension_months": "Conditional Extension (Months)"})
overview["Listed Price"] = overview["Listed Price"].map("{:,.2f}".format)
show_table(overview)
display(HTML("""
<style>
.report-table th,
.report-table td {
    text-align: center !important;
}
</style>
"""))

Exam Prep Course,Listed Price,Currency,Access (Months),Original Practice Exams,Conditional Extension (Months)
CAS Exam 5,500.00,USD,7,4,6
CAS Exam 6C,"1,000.00",CAD,7,1,6
CAS Exam 6U,"1,000.00",USD,7,2,6
CAS Exam 7,"1,000.00",USD,7,2,6
CAS Exam 8,"1,000.00",USD,7,2,6
CAS Exam 9,"1,000.00",USD,7,2,6


### Table 2. Dataset Completeness

PDF and Excel measurements are stored separately so each dataset holds only relevant columns. A failed or empty extraction would appear in this table.

In [21]:
def completeness(name, frame, key):
    blank = frame.apply(lambda col: col.map(lambda v: isinstance(v, str) and not v.strip()))
    failed = int(frame["extraction_status"].ne("Extracted").sum()) if "extraction_status" in frame else 0
    return {"Dataset": name, "Observations": len(frame),
            "Missing or Blank Cells": int((frame.isna() | blank).sum().sum()),
            "Duplicate Source URLs": int(frame[key].duplicated().sum()),
            "Failed Extractions": failed}


show_table(pd.DataFrame([
    completeness("Products", products_final, "product_url"),
    completeness("PDF samples", pdf_samples, "sample_url"),
    completeness("Excel samples", excel_samples, "sample_url")]))
display(HTML("""
<style>
.report-table th,
.report-table td {
    text-align: center !important;
}
</style>
"""))

Dataset,Observations,Missing or Blank Cells,Duplicate Source URLs,Failed Extractions
Products,6,0,0,0
PDF samples,9,0,0,0
Excel samples,6,0,0,0


## Data Quality

The collection contains six products and 25 advertised sample cards. Sixteen cards link to 15 unique non-video files: nine PDFs and six Excel workbooks. Two Exam 5 cards share one PDF, so their labels are kept together in one record. The eight video cards were excluded from downloads. One Exam 9 card is marked "Coming Soon" and has no URL, so it is not counted as an available file.

All 15 files yielded text. Excel text covers every worksheet; worksheet counts include instruction and contents sheets, not only practice questions. Sample size and structure do not measure course quality or coverage.

Checks in the code confirm six unique product URLs, no missing core product fields, unique file URLs, that every file links to a product, and that the file count matches the unique linked URLs. Exam 5's price, access period, conditional extension, and practice-exam count were also compared with my manual collection. Currency is stored separately from the numeric price.

### Table 3. Numeric Summaries

Variances are sample variances. Price variances are in squared currency units. The CAD variance is undefined with one observation.

In [20]:
rows = []


def summarize(label, values):
    rows.append({"Measure": label, "Observations": values.count(), "Missing": values.isna().sum(),
                 "Mean": values.mean(), "Sample Variance": values.var()})


for currency, group in products_final.groupby("currency"):  # prices are never mixed across currencies
    summarize(f"Price ({currency})", group["price"])
for col, label in {"duration_months": "Standard access (months)",
                   "practice_exam_count": "Original practice exams",
                   "free_extension_months": "Conditional extension (months)"}.items():
    summarize(label, products_final[col])
show_table(pd.DataFrame(rows).round(2))
display(HTML("""
<style>
.report-table th,
.report-table td {
    text-align: center !important;
}
</style>
"""))

Measure,Observations,Missing,Mean,Sample Variance
Price (CAD),1,0,1000.00,—
Price (USD),5,0,900.00,50000.00
Standard access (months),6,0,7.00,0.00
Original practice exams,6,0,2.17,0.97
Conditional extension (months),6,0,6.00,0.00


### Table 4. Public PDF and Excel Samples by Course

Counts refer to unique files, not advertised cards. Page and worksheet ranges describe the available samples, not the size or quality of the paid course.

In [19]:
def file_counts(values):
    values = values.dropna().sort_values()
    return ", ".join(str(int(value)) for value in values) if len(values) else "0"

rows = []
for _, p in products_final.iterrows():
    pdfs = pdf_samples[pdf_samples["product_url"].eq(p["product_url"])]
    books = excel_samples[excel_samples["product_url"].eq(p["product_url"])]
    rows.append({
        "Exam Prep Course": p["product_name"],
        "PDF Files": len(pdfs),
        "PDF Pages (Per File)": file_counts(pdfs["page_count"]),
        "Excel Files": len(books),
        "Worksheets (Per File)": file_counts(books["sheet_count"])
    })

show_table(pd.DataFrame(rows))

display(HTML("""
<style>
.report-table th,
.report-table td {
    text-align: center !important;
}
</style>
"""))

Exam Prep Course,PDF Files,PDF Pages (Per File),Excel Files,Worksheets (Per File)
CAS Exam 5,2,"2, 17",1,32
CAS Exam 6C,1,7,1,11
CAS Exam 6U,2,"2, 5",2,"3, 29"
CAS Exam 7,2,"2, 9",1,4
CAS Exam 8,1,6,1,16
CAS Exam 9,1,7,0,0


## Discussion

Five courses are priced in USD, averaging $900; Exam 6C is listed separately at CAD 1,000. All six list seven months of standard access and a conditional six-month extension. Original practice-exam counts range from one to four.

Mentions of instructional videos, formula/summary sheets, and hybrid exams differ across courses. The 25 sample cards include 24 with listed destinations and one "Coming Soon" card. Text was extracted from all 15 unique linked PDF and Excel files (16 cards), ready for later content comparisons.

These data describe one provider at one point in time. Advertised features and sample excerpts do not measure course quality, exam difficulty, or student outcomes, and sample sizes and document structures are not comparable measures of coverage.

## Export Final Datasets

The submission datasets are Products, PDF Samples, and Excel Samples.

In [12]:
products_final.to_csv("battleacts_products.csv", index=False)
pdf_samples.to_csv("battleacts_pdf_samples.csv", index=False)
excel_samples.to_csv("battleacts_excel_samples.csv", index=False)